In [5]:
%pip install mediapipe opencv-python tqdm numpy pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import mediapipe as mp

In [14]:
PROCESSED_FRAMES_DIR = Path("../Dataset/processed_frames")

FRAMES_DECEPTIVE_DIR = PROCESSED_FRAMES_DIR / "deceptive"
FRAMES_TRUTHFUL_DIR = PROCESSED_FRAMES_DIR / "truthful"

LANDMARKS_OUTPUT_DIR = Path("processed_landmarks")

LANDMARKS_DECEPTIVE_DIR = LANDMARKS_OUTPUT_DIR / "deceptive"
LANDMARKS_TRUTHFUL_DIR = LANDMARKS_OUTPUT_DIR / "truthful"

LANDMARKS_DECEPTIVE_DIR.mkdir(parents=True, exist_ok=True)
LANDMARKS_TRUTHFUL_DIR.mkdir(parents=True, exist_ok=True)

In [8]:
# Cell 4: Download the MediaPipe Face Landmarker model bundle

from pathlib import Path
import urllib.request

MODEL_DIR = Path("mediapipe_models")
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FACE_LANDMARKER_MODEL_PATH = MODEL_DIR / "face_landmarker.task"

if not FACE_LANDMARKER_MODEL_PATH.exists():
    url = "https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task"
    urllib.request.urlretrieve(url, FACE_LANDMARKER_MODEL_PATH)
    print("Downloaded:", FACE_LANDMARKER_MODEL_PATH)
else:
    print("Model already exists:", FACE_LANDMARKER_MODEL_PATH)

Downloaded: mediapipe_models\face_landmarker.task


In [9]:
# Cell 5: Initialize MediaPipe Tasks API FaceLandmarker

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

BaseOptions = python.BaseOptions
FaceLandmarker = vision.FaceLandmarker
FaceLandmarkerOptions = vision.FaceLandmarkerOptions
VisionRunningMode = vision.RunningMode

options = FaceLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=str(FACE_LANDMARKER_MODEL_PATH)
    ),
    running_mode=VisionRunningMode.IMAGE,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False
)

landmarker = FaceLandmarker.create_from_options(options)

print("MediaPipe Tasks FaceLandmarker initialized successfully.")

MediaPipe Tasks FaceLandmarker initialized successfully.


In [10]:
# Cell 6: Extract landmarks from a single frame using MediaPipe Tasks API

import cv2
import numpy as np

def extract_landmarks_from_frame(frame_path):
    """
    Extract facial landmarks from one image frame using MediaPipe Tasks API.

    Args:
        frame_path: Path to the image frame.

    Returns:
        numpy array of shape (N, 3), or None if no face is detected.
        For the current FaceLandmarker model, N is usually 478 landmarks.
    """

    frame_path = Path(frame_path)

    image_bgr = cv2.imread(str(frame_path))

    if image_bgr is None:
        return None

    # OpenCV reads BGR, MediaPipe expects SRGB/RGB.
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    # Make sure the array is contiguous and uint8.
    image_rgb = np.ascontiguousarray(image_rgb, dtype=np.uint8)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=image_rgb
    )

    result = landmarker.detect(mp_image)

    if not result.face_landmarks:
        return None

    # We requested num_faces=1, so use the first detected face.
    face_landmarks = result.face_landmarks[0]

    landmarks = []

    for landmark in face_landmarks:
        landmarks.append([
            landmark.x,
            landmark.y,
            landmark.z
        ])

    return np.array(landmarks, dtype=np.float32)

In [11]:
# Cell 7: Normalize landmarks

def normalize_landmarks(landmarks):
    """
    Normalize facial landmarks to reduce position and scale differences.

    Args:
        landmarks: numpy array of shape (N, 3)

    Returns:
        flattened normalized vector of shape (N * 3,)
    """

    landmarks = np.array(landmarks, dtype=np.float32)

    # Center the face around the mean landmark position.
    center = np.mean(landmarks, axis=0)
    centered_landmarks = landmarks - center

    # Scale normalization.
    min_vals = np.min(centered_landmarks, axis=0)
    max_vals = np.max(centered_landmarks, axis=0)

    face_size = np.linalg.norm(max_vals - min_vals)

    if face_size == 0:
        return None

    normalized_landmarks = centered_landmarks / face_size

    # Flatten from (N, 3) to (N * 3,)
    flattened = normalized_landmarks.flatten()

    return flattened.astype(np.float32)

In [15]:
# Cell 8: Test landmark extraction on one sample frame

sample_deceptive_folders = sorted([
    folder for folder in FRAMES_DECEPTIVE_DIR.iterdir()
    if folder.is_dir()
])

sample_folder = sample_deceptive_folders[0]
sample_frames = sorted(list(sample_folder.glob("*.jpg")))

sample_frame = sample_frames[0]

landmarks = extract_landmarks_from_frame(sample_frame)

print("Sample frame:", sample_frame)

if landmarks is None:
    print("No face detected.")
else:
    print("Raw landmark shape:", landmarks.shape)

    normalized = normalize_landmarks(landmarks)
    print("Normalized vector shape:", normalized.shape)

Sample frame: ..\Dataset\processed_frames\deceptive\trial_lie_001\frame_000000.jpg
No face detected.


In [16]:
# Cell 9: Convert all frames from one video into one landmark sequence

def process_video_frame_folder(frame_folder, output_file):
    """
    Converts all frames of one video into a normalized landmark sequence.

    Args:
        frame_folder: folder containing extracted frames for one video
        output_file: where to save the .npy landmark sequence

    Returns:
        dictionary summary
    """

    frame_folder = Path(frame_folder)
    output_file = Path(output_file)

    frame_paths = sorted(list(frame_folder.glob("*.jpg")))

    landmark_sequence = []
    missing_frames = 0

    for frame_path in frame_paths:
        landmarks = extract_landmarks_from_frame(frame_path)

        if landmarks is None:
            missing_frames += 1
            continue

        normalized = normalize_landmarks(landmarks)

        if normalized is None:
            missing_frames += 1
            continue

        landmark_sequence.append(normalized)

    if len(landmark_sequence) == 0:
        return {
            "video": frame_folder.name,
            "status": "failed",
            "total_frames": len(frame_paths),
            "valid_frames": 0,
            "missing_frames": missing_frames,
            "feature_dim": None,
            "output_file": None
        }

    landmark_sequence = np.array(landmark_sequence, dtype=np.float32)

    np.save(output_file, landmark_sequence)

    return {
        "video": frame_folder.name,
        "status": "success",
        "total_frames": len(frame_paths),
        "valid_frames": landmark_sequence.shape[0],
        "missing_frames": missing_frames,
        "feature_dim": landmark_sequence.shape[1],
        "output_file": str(output_file)
    }

In [17]:
# Cell 10: Process deceptive frame folders

deceptive_frame_folders = sorted([
    folder for folder in FRAMES_DECEPTIVE_DIR.iterdir()
    if folder.is_dir()
])

deceptive_landmark_results = []

for frame_folder in tqdm(deceptive_frame_folders, desc="Processing deceptive landmarks"):
    output_file = LANDMARKS_DECEPTIVE_DIR / f"{frame_folder.name}.npy"

    result = process_video_frame_folder(
        frame_folder=frame_folder,
        output_file=output_file
    )

    deceptive_landmark_results.append(result)

print("Finished deceptive landmark extraction.")

Processing deceptive landmarks: 100%|██████████| 61/61 [10:18<00:00, 10.13s/it]

Finished deceptive landmark extraction.


In [18]:
# Cell 11: Process truthful frame folders

truthful_frame_folders = sorted([
    folder for folder in FRAMES_TRUTHFUL_DIR.iterdir()
    if folder.is_dir()
])

truthful_landmark_results = []

for frame_folder in tqdm(truthful_frame_folders, desc="Processing truthful landmarks"):
    output_file = LANDMARKS_TRUTHFUL_DIR / f"{frame_folder.name}.npy"

    result = process_video_frame_folder(
        frame_folder=frame_folder,
        output_file=output_file
    )

    truthful_landmark_results.append(result)

print("Finished truthful landmark extraction.")

Processing truthful landmarks: 100%|██████████| 60/60 [06:52<00:00,  6.87s/it]

Finished truthful landmark extraction.


In [19]:
# Cell 12: Save extraction log

all_landmark_results = deceptive_landmark_results + truthful_landmark_results

df_landmarks = pd.DataFrame(all_landmark_results)

log_path = LANDMARKS_OUTPUT_DIR / "landmark_extraction_log.csv"
df_landmarks.to_csv(log_path, index=False)

df_landmarks.head()

,video,status,total_frames,valid_frames,missing_frames,feature_dim,output_file
0,trial_lie_001,success,85,11,74,1434.0,processed_landmarks\deceptive\trial_lie_001.npy
1,trial_lie_002,success,313,196,117,1434.0,processed_landmarks\deceptive\trial_lie_002.npy
2,trial_lie_003,success,36,8,28,1434.0,processed_landmarks\deceptive\trial_lie_003.npy
3,trial_lie_004,success,59,22,37,1434.0,processed_landmarks\deceptive\trial_lie_004.npy
4,trial_lie_005,success,267,44,223,1434.0,processed_landmarks\deceptive\trial_lie_005.npy


In [20]:
# Cell 13: Check extraction summary

print("Total videos processed:", len(df_landmarks))
print("Successful:", len(df_landmarks[df_landmarks["status"] == "success"]))
print("Failed:", len(df_landmarks[df_landmarks["status"] == "failed"]))

if len(df_landmarks[df_landmarks["status"] == "success"]) > 0:
    print("\nAverage valid frames:")
    print(df_landmarks["valid_frames"].mean())

    print("\nAverage missing frames:")
    print(df_landmarks["missing_frames"].mean())

    print("\nFeature dimensions found:")
    print(df_landmarks["feature_dim"].value_counts(dropna=False))

Total videos processed: 121
Successful: 113
Failed: 8

Average valid frames:
112.39669421487604

Average missing frames:
28.33884297520661

Feature dimensions found:
feature_dim
1434.0    113
NaN         8
Name: count, dtype: int64


In [21]:
# Cell 14: Inspect one saved .npy file

sample_files = sorted(list(LANDMARKS_DECEPTIVE_DIR.glob("*.npy")))

if len(sample_files) == 0:
    print("No landmark files found.")
else:
    sample_file = sample_files[0]
    sample_sequence = np.load(sample_file)

    print("Sample file:", sample_file)
    print("Sequence shape:", sample_sequence.shape)
    print("Data type:", sample_sequence.dtype)

Sample file: processed_landmarks\deceptive\trial_lie_001.npy
Sequence shape: (11, 1434)
Data type: float32


In [22]:
# Cell 15: Calculate missing frame percentage

df_landmarks["missing_percentage"] = (
    df_landmarks["missing_frames"] / df_landmarks["total_frames"]
) * 100

df_landmarks[
    ["video", "status", "total_frames", "valid_frames", "missing_frames", "missing_percentage", "feature_dim"]
].head()

,video,status,total_frames,valid_frames,missing_frames,missing_percentage,feature_dim
0,trial_lie_001,success,85,11,74,87.058824,1434.0
1,trial_lie_002,success,313,196,117,37.380192,1434.0
2,trial_lie_003,success,36,8,28,77.777778,1434.0
3,trial_lie_004,success,59,22,37,62.711864,1434.0
4,trial_lie_005,success,267,44,223,83.520599,1434.0


In [23]:
# Cell 16: Create final dataset CSV for model training

data_rows = []

for file_path in sorted(LANDMARKS_DECEPTIVE_DIR.glob("*.npy")):
    data_rows.append({
        "landmark_path": str(file_path),
        "video_name": file_path.stem,
        "label": 1
    })

for file_path in sorted(LANDMARKS_TRUTHFUL_DIR.glob("*.npy")):
    data_rows.append({
        "landmark_path": str(file_path),
        "video_name": file_path.stem,
        "label": 0
    })

df_dataset = pd.DataFrame(data_rows)

dataset_csv_path = LANDMARKS_OUTPUT_DIR / "video_landmark_dataset.csv"
df_dataset.to_csv(dataset_csv_path, index=False)

print("Dataset CSV saved to:", dataset_csv_path)
print("Total landmark sequences:", len(df_dataset))

df_dataset.head()

Dataset CSV saved to: processed_landmarks\video_landmark_dataset.csv
Total landmark sequences: 113


,landmark_path,video_name,label
0,processed_landmarks\deceptive\trial_lie_001.npy,trial_lie_001,1
1,processed_landmarks\deceptive\trial_lie_002.npy,trial_lie_002,1
2,processed_landmarks\deceptive\trial_lie_003.npy,trial_lie_003,1
3,processed_landmarks\deceptive\trial_lie_004.npy,trial_lie_004,1
4,processed_landmarks\deceptive\trial_lie_005.npy,trial_lie_005,1
